# Which specific columns were rejected, and on what number?

*Setup cells below are carried from the shared analysis so this notebook runs on its own.*


In [9]:
import json
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Path to the materialized contract
contract_path = Path("../../references/feature-contract.json")

# Fallback structure if the contract hasn't been generated yet
if not contract_path.exists():
    print(f"⚠️ Warning: The contract file was not found at `{contract_path}`.")
    print("Run the `feature_contract` asset in Dagster to generate it first.")
    
else:
    with open(contract_path, "r") as f:
        contract_data = json.load(f)
        
    summary = contract_data.get("summary", {})
    fragments = contract_data.get("fragments", [])
    columns = contract_data.get("columns", [])

# Extract counts per check
rejections_by_check = {f["check"]: f["rejected"] for f in fragments}

In [7]:
from IPython.display import Markdown, display

table_md = f"""
| Metric | Count |
|--------|-------|
| **Declared Features** | {summary.get('declared', 0)} |
| **Admitted to Model** | {summary.get('admitted', 0)} |
| **Rejected by Audits** | {summary.get('rejected', 0)} |
| **Manual Overrides** | {summary.get('overridden', 0)} |
"""
display(Markdown(table_md))


| Metric | Count |
|--------|-------|
| **Declared Features** | 479 |
| **Admitted to Model** | 208 |
| **Rejected by Audits** | 271 |
| **Manual Overrides** | 0 |


In [8]:
# Prepare data for waterfall chart
steps = ["Declared"]
measures = ["absolute"]
values = [summary.get("declared", 0)]

for check, count in rejections_by_check.items():
    if count > 0:
        steps.append(f"Rejected: {check.replace('_', ' ').title()}")
        measures.append("relative")
        values.append(-count)

if summary.get("overridden", 0) > 0:
    steps.append("Manual Overrides")
    measures.append("relative")
    values.append(summary.get("overridden", 0))

steps.append("Admitted")
measures.append("total")
values.append(summary.get("admitted", 0))

fig = go.Figure(go.Waterfall(
    name="Feature Contract",
    orientation="v",
    measure=measures,
    x=steps,
    textposition="outside",
    text=[str(v) if m != "total" else str(v) for m, v in zip(measures, values)],
    y=values,
    connector={"line":{"color":"rgb(63, 63, 63)"}},
    decreasing={"marker":{"color":"#ef553b"}},
    increasing={"marker":{"color":"#00cc96"}},
    totals={"marker":{"color":"#636efa"}}
))

fig.update_layout(
    title="Feature Admittance Funnel",
    showlegend=False,
    plot_bgcolor='rgba(0,0,0,0)',
    yaxis_title="Number of Features",
    height=500
)

fig.show()

In [4]:
# Prepare data for bar chart
if sum(rejections_by_check.values()) > 0:
    df_rejections = pd.DataFrame(list(rejections_by_check.items()), columns=["Audit Step", "Rejected Count"])
    df_rejections["Audit Step"] = df_rejections["Audit Step"].str.replace("_", " ").str.title()
    df_rejections = df_rejections[df_rejections["Rejected Count"] > 0].sort_values("Rejected Count", ascending=True)

    fig_bar = px.bar(
        df_rejections, 
        x="Rejected Count", 
        y="Audit Step", 
        orientation='h',
        title="Why were features rejected?",
        color="Rejected Count",
        color_continuous_scale="Reds"
    )

    fig_bar.update_layout(
        plot_bgcolor='rgba(0,0,0,0)',
        xaxis_title="Number of Rejected Features",
        yaxis_title="",
        coloraxis_showscale=False
    )

    fig_bar.show()
else:
    print("No features were rejected.")

## Column-Level Rejections

The table below provides a detailed breakdown of every feature that was rejected, along with the exact audit step that caught it, and the numerical score that exceeded the policy threshold.

In [5]:
from IPython.display import Markdown, display

rejected_cols = [c for c in columns if not c.get("admitted", True)]

if rejected_cols:
    df_cols = pd.DataFrame(rejected_cols)[["name", "rejected_by", "rejected_value", "rejected_unit"]]
    df_cols = df_cols.sort_values(["rejected_by", "name"])
    
    # Format the value column to 4 decimal places if it's a float
    df_cols["rejected_value"] = df_cols["rejected_value"].apply(lambda x: f"{x:.4f}" if isinstance(x, float) else x)
    
    # Rename columns for presentation
    df_cols.columns = ["Feature Name", "Rejected By", "Value / Score", "Unit / Block"]
    
    # Display as a markdown table
    display(Markdown(df_cols.to_markdown(index=False)))
else:
    print("No features were rejected.")

| Feature Name                  | Rejected By        |   Value / Score | Unit / Block    |
|:------------------------------|:-------------------|----------------:|:----------------|
| D10n                          | distribution_shift |          2.31   | column          |
| D11                           | distribution_shift |          0.8358 | column          |
| D11n                          | distribution_shift |          1.7807 | column          |
| D15n                          | distribution_shift |          1.9497 | column          |
| D1n                           | distribution_shift |          3.0916 | column          |
| D2n                           | distribution_shift |          0.8343 | column          |
| D5n                           | distribution_shift |          2.2003 | column          |
| M1                            | distribution_shift |          0.6919 | column          |
| M2                            | distribution_shift |          0.6937 | column          |
| M3                            | distribution_shift |          0.6919 | column          |
| M7                            | distribution_shift |          0.9587 | column          |
| M8                            | distribution_shift |          0.9596 | column          |
| M9                            | distribution_shift |          0.9604 | column          |
| V10                           | distribution_shift |          0.7866 | column          |
| V11                           | distribution_shift |          0.7867 | column          |
| V144                          | distribution_shift |          0.7684 | column          |
| V145                          | distribution_shift |          0.8592 | column          |
| V150                          | distribution_shift |          0.8394 | column          |
| V151                          | distribution_shift |          0.8417 | column          |
| V152                          | distribution_shift |          0.7908 | column          |
| V2                            | distribution_shift |          0.7993 | column          |
| V3                            | distribution_shift |          0.7985 | column          |
| V4                            | distribution_shift |          0.7925 | column          |
| V5                            | distribution_shift |          0.7895 | column          |
| V6                            | distribution_shift |          0.7961 | column          |
| V7                            | distribution_shift |          0.7942 | column          |
| V8                            | distribution_shift |          0.796  | column          |
| V9                            | distribution_shift |          0.7945 | column          |
| client_d3n_std_prior          | distribution_shift |          0.9129 | column          |
| client_m8_mean_prior          | distribution_shift |          0.9144 | column          |
| client_txn_count_prior        | distribution_shift |          0.5521 | column          |
| device_txn_count_24h          | distribution_shift |          0.8891 | column          |
| id_13                         | distribution_shift |          0.6975 | column          |
| id_31                         | distribution_shift |          1.3828 | column          |
| null_count_V_block            | distribution_shift |          0.9134 | column          |
| seconds_since_prev_txn_client | distribution_shift |          0.5974 | column          |
| V100                          | redundancy         |        nan      | block:V95-V106  |
| V101                          | redundancy         |        nan      | block:V95-V106  |
| V102                          | redundancy         |        nan      | block:V95-V106  |
| V103                          | redundancy         |        nan      | block:V95-V106  |
| V105                          | redundancy         |        nan      | block:V95-V106  |
| V106                          | redundancy         |        nan      | block:V95-V106  |
| V109                          | redundancy         |        nan      | block:V107-V123 |
| V110                          | redundancy         |        nan      | block:V107-V123 |
| V112                          | redundancy         |        nan      | block:V107-V123 |
| V113                          | redundancy         |        nan      | block:V107-V123 |
| V114                          | redundancy         |        nan      | block:V107-V123 |
| V116                          | redundancy         |        nan      | block:V107-V123 |
| V118                          | redundancy         |        nan      | block:V107-V123 |
| V119                          | redundancy         |        nan      | block:V107-V123 |
| V12                           | redundancy         |        nan      | block:V12-V34   |
| V122                          | redundancy         |        nan      | block:V107-V123 |
| V125                          | redundancy         |        nan      | block:V124-V137 |
| V126                          | redundancy         |        nan      | block:V124-V137 |
| V128                          | redundancy         |        nan      | block:V124-V137 |
| V131                          | redundancy         |        nan      | block:V124-V137 |
| V132                          | redundancy         |        nan      | block:V124-V137 |
| V133                          | redundancy         |        nan      | block:V124-V137 |
| V134                          | redundancy         |        nan      | block:V124-V137 |
| V135                          | redundancy         |        nan      | block:V124-V137 |
| V137                          | redundancy         |        nan      | block:V124-V137 |
| V140                          | redundancy         |        nan      | block:V138-V163 |
| V143                          | redundancy         |        nan      | block:V143-V166 |
| V146                          | redundancy         |        nan      | block:V138-V163 |
| V148                          | redundancy         |        nan      | block:V138-V163 |
| V149                          | redundancy         |        nan      | block:V138-V163 |
| V15                           | redundancy         |        nan      | block:V12-V34   |
| V153                          | redundancy         |        nan      | block:V138-V163 |
| V154                          | redundancy         |        nan      | block:V138-V163 |
| V157                          | redundancy         |        nan      | block:V138-V163 |
| V158                          | redundancy         |        nan      | block:V138-V163 |
| V16                           | redundancy         |        nan      | block:V12-V34   |
| V164                          | redundancy         |        nan      | block:V143-V166 |
| V167                          | redundancy         |        nan      | block:V167-V183 |
| V168                          | redundancy         |        nan      | block:V167-V183 |
| V170                          | redundancy         |        nan      | block:V169-V210 |
| V172                          | redundancy         |        nan      | block:V167-V183 |
| V174                          | redundancy         |        nan      | block:V169-V210 |
| V177                          | redundancy         |        nan      | block:V167-V183 |
| V179                          | redundancy         |        nan      | block:V167-V183 |
| V18                           | redundancy         |        nan      | block:V12-V34   |
| V181                          | redundancy         |        nan      | block:V167-V183 |
| V183                          | redundancy         |        nan      | block:V167-V183 |
| V184                          | redundancy         |        nan      | block:V169-V210 |
| V186                          | redundancy         |        nan      | block:V186-V216 |
| V189                          | redundancy         |        nan      | block:V169-V210 |
| V19                           | redundancy         |        nan      | block:V12-V34   |
| V190                          | redundancy         |        nan      | block:V186-V216 |
| V191                          | redundancy         |        nan      | block:V186-V216 |
| V192                          | redundancy         |        nan      | block:V186-V216 |
| V193                          | redundancy         |        nan      | block:V186-V216 |
| V194                          | redundancy         |        nan      | block:V169-V210 |
| V195                          | redundancy         |        nan      | block:V169-V210 |
| V196                          | redundancy         |        nan      | block:V186-V216 |
| V197                          | redundancy         |        nan      | block:V169-V210 |
| V199                          | redundancy         |        nan      | block:V186-V216 |
| V200                          | redundancy         |        nan      | block:V169-V210 |
| V201                          | redundancy         |        nan      | block:V169-V210 |
| V202                          | redundancy         |        nan      | block:V186-V216 |
| V204                          | redundancy         |        nan      | block:V186-V216 |
| V206                          | redundancy         |        nan      | block:V186-V216 |
| V208                          | redundancy         |        nan      | block:V169-V210 |
| V21                           | redundancy         |        nan      | block:V12-V34   |
| V211                          | redundancy         |        nan      | block:V186-V216 |
| V212                          | redundancy         |        nan      | block:V186-V216 |
| V213                          | redundancy         |        nan      | block:V186-V216 |
| V214                          | redundancy         |        nan      | block:V186-V216 |
| V216                          | redundancy         |        nan      | block:V186-V216 |
| V217                          | redundancy         |        nan      | block:V217-V237 |
| V219                          | redundancy         |        nan      | block:V217-V237 |
| V22                           | redundancy         |        nan      | block:V12-V34   |
| V222                          | redundancy         |        nan      | block:V220-V272 |
| V225                          | redundancy         |        nan      | block:V217-V237 |
| V227                          | redundancy         |        nan      | block:V220-V272 |
| V230                          | redundancy         |        nan      | block:V217-V237 |
| V231                          | redundancy         |        nan      | block:V217-V237 |
| V232                          | redundancy         |        nan      | block:V217-V237 |
| V233                          | redundancy         |        nan      | block:V217-V237 |
| V236                          | redundancy         |        nan      | block:V217-V237 |
| V237                          | redundancy         |        nan      | block:V217-V237 |
| V239                          | redundancy         |        nan      | block:V220-V272 |
| V24                           | redundancy         |        nan      | block:V12-V34   |
| V241                          | redundancy         |        nan      | block:V240-V262 |
| V242                          | redundancy         |        nan      | block:V240-V262 |
| V243                          | redundancy         |        nan      | block:V240-V262 |
| V244                          | redundancy         |        nan      | block:V240-V262 |
| V245                          | redundancy         |        nan      | block:V220-V272 |
| V246                          | redundancy         |        nan      | block:V240-V262 |
| V247                          | redundancy         |        nan      | block:V240-V262 |
| V248                          | redundancy         |        nan      | block:V240-V262 |
| V249                          | redundancy         |        nan      | block:V240-V262 |
| V251                          | redundancy         |        nan      | block:V220-V272 |
| V254                          | redundancy         |        nan      | block:V240-V262 |
| V255                          | redundancy         |        nan      | block:V220-V272 |
| V256                          | redundancy         |        nan      | block:V220-V272 |
| V259                          | redundancy         |        nan      | block:V220-V272 |
| V262                          | redundancy         |        nan      | block:V240-V262 |
| V263                          | redundancy         |        nan      | block:V263-V278 |
| V265                          | redundancy         |        nan      | block:V263-V278 |
| V268                          | redundancy         |        nan      | block:V263-V278 |
| V269                          | redundancy         |        nan      | block:V263-V278 |
| V270                          | redundancy         |        nan      | block:V220-V272 |
| V272                          | redundancy         |        nan      | block:V220-V272 |
| V273                          | redundancy         |        nan      | block:V263-V278 |
| V275                          | redundancy         |        nan      | block:V263-V278 |
| V276                          | redundancy         |        nan      | block:V263-V278 |
| V278                          | redundancy         |        nan      | block:V263-V278 |
| V279                          | redundancy         |        nan      | block:V279-V299 |
| V28                           | redundancy         |        nan      | block:V12-V34   |
| V280                          | redundancy         |        nan      | block:V279-V299 |
| V282                          | redundancy         |        nan      | block:V281-V315 |
| V287                          | redundancy         |        nan      | block:V279-V299 |
| V288                          | redundancy         |        nan      | block:V281-V315 |
| V29                           | redundancy         |        nan      | block:V12-V34   |
| V290                          | redundancy         |        nan      | block:V279-V299 |
| V292                          | redundancy         |        nan      | block:V279-V299 |
| V293                          | redundancy         |        nan      | block:V279-V299 |
| V295                          | redundancy         |        nan      | block:V279-V299 |
| V298                          | redundancy         |        nan      | block:V279-V299 |
| V299                          | redundancy         |        nan      | block:V279-V299 |
| V300                          | redundancy         |        nan      | block:V281-V315 |
| V302                          | redundancy         |        nan      | block:V302-V321 |
| V304                          | redundancy         |        nan      | block:V302-V321 |
| V306                          | redundancy         |        nan      | block:V302-V321 |
| V308                          | redundancy         |        nan      | block:V302-V321 |
| V31                           | redundancy         |        nan      | block:V12-V34   |
| V311                          | redundancy         |        nan      | block:V302-V321 |
| V312                          | redundancy         |        nan      | block:V302-V321 |
| V313                          | redundancy         |        nan      | block:V281-V315 |
| V315                          | redundancy         |        nan      | block:V281-V315 |
| V316                          | redundancy         |        nan      | block:V302-V321 |
| V317                          | redundancy         |        nan      | block:V302-V321 |
| V318                          | redundancy         |        nan      | block:V302-V321 |
| V319                          | redundancy         |        nan      | block:V302-V321 |
| V32                           | redundancy         |        nan      | block:V12-V34   |
| V321                          | redundancy         |        nan      | block:V302-V321 |
| V322                          | redundancy         |        nan      | block:V322-V339 |
| V323                          | redundancy         |        nan      | block:V322-V339 |
| V324                          | redundancy         |        nan      | block:V322-V339 |
| V33                           | redundancy         |        nan      | block:V12-V34   |
| V331                          | redundancy         |        nan      | block:V322-V339 |
| V333                          | redundancy         |        nan      | block:V322-V339 |
| V34                           | redundancy         |        nan      | block:V12-V34   |
| V35                           | redundancy         |        nan      | block:V35-V52   |
| V38                           | redundancy         |        nan      | block:V35-V52   |
| V39                           | redundancy         |        nan      | block:V35-V52   |
| V42                           | redundancy         |        nan      | block:V35-V52   |
| V43                           | redundancy         |        nan      | block:V35-V52   |
| V45                           | redundancy         |        nan      | block:V35-V52   |
| V46                           | redundancy         |        nan      | block:V35-V52   |
| V49                           | redundancy         |        nan      | block:V35-V52   |
| V50                           | redundancy         |        nan      | block:V35-V52   |
| V51                           | redundancy         |        nan      | block:V35-V52   |
| V52                           | redundancy         |        nan      | block:V35-V52   |
| V53                           | redundancy         |        nan      | block:V53-V74   |
| V57                           | redundancy         |        nan      | block:V53-V74   |
| V58                           | redundancy         |        nan      | block:V53-V74   |
| V60                           | redundancy         |        nan      | block:V53-V74   |
| V63                           | redundancy         |        nan      | block:V53-V74   |
| V64                           | redundancy         |        nan      | block:V53-V74   |
| V69                           | redundancy         |        nan      | block:V53-V74   |
| V71                           | redundancy         |        nan      | block:V53-V74   |
| V72                           | redundancy         |        nan      | block:V53-V74   |
| V73                           | redundancy         |        nan      | block:V53-V74   |
| V74                           | redundancy         |        nan      | block:V53-V74   |
| V75                           | redundancy         |        nan      | block:V75-V94   |
| V77                           | redundancy         |        nan      | block:V75-V94   |
| V79                           | redundancy         |        nan      | block:V75-V94   |
| V81                           | redundancy         |        nan      | block:V75-V94   |
| V83                           | redundancy         |        nan      | block:V75-V94   |
| V84                           | redundancy         |        nan      | block:V75-V94   |
| V85                           | redundancy         |        nan      | block:V75-V94   |
| V87                           | redundancy         |        nan      | block:V75-V94   |
| V90                           | redundancy         |        nan      | block:V75-V94   |
| V92                           | redundancy         |        nan      | block:V75-V94   |
| V93                           | redundancy         |        nan      | block:V75-V94   |
| V94                           | redundancy         |        nan      | block:V75-V94   |
| V95                           | redundancy         |        nan      | block:V95-V106  |
| V97                           | redundancy         |        nan      | block:V95-V106  |
| D3n                           | time_consistency   |         -0.1681 | column          |
| V138                          | time_consistency   |         -0.0984 | column          |
| V141                          | time_consistency   |         -0.1045 | column          |
| V142                          | time_consistency   |         -0.1044 | column          |
| V159                          | time_consistency   |         -0.1122 | column          |
| V160                          | time_consistency   |         -0.114  | column          |
| V161                          | time_consistency   |         -0.1135 | column          |
| V162                          | time_consistency   |         -0.1096 | column          |
| V163                          | time_consistency   |         -0.1129 | column          |
| V25                           | time_consistency   |         -0.0685 | column          |
| V26                           | time_consistency   |         -0.0687 | column          |
| V325                          | time_consistency   |         -0.0972 | column          |
| V326                          | time_consistency   |         -0.088  | column          |
| V327                          | time_consistency   |         -0.0881 | column          |
| V328                          | time_consistency   |         -0.0922 | column          |
| V329                          | time_consistency   |         -0.0889 | column          |
| V330                          | time_consistency   |         -0.0922 | column          |
| V334                          | time_consistency   |         -0.1026 | column          |
| V335                          | time_consistency   |         -0.104  | column          |
| V336                          | time_consistency   |         -0.0993 | column          |
| V337                          | time_consistency   |         -0.0939 | column          |
| V338                          | time_consistency   |         -0.0943 | column          |
| V339                          | time_consistency   |         -0.0939 | column          |
| V55                           | time_consistency   |         -0.0718 | column          |
| V61                           | time_consistency   |         -0.0647 | column          |
| V65                           | time_consistency   |         -0.0828 | column          |
| V66                           | time_consistency   |         -0.0802 | column          |
| V67                           | time_consistency   |         -0.0823 | column          |
| V68                           | time_consistency   |         -0.0829 | column          |
| V88                           | time_consistency   |         -0.0642 | column          |
| V89                           | time_consistency   |         -0.0643 | column          |
| client_amt_deviation_prior    | time_consistency   |         -0.1206 | column          |
| client_c10_mean_prior         | time_consistency   |         -0.1624 | column          |
| client_c11_mean_prior         | time_consistency   |         -0.1334 | column          |
| client_c12_mean_prior         | time_consistency   |         -0.1118 | column          |
| client_c13_mean_prior         | time_consistency   |         -0.1331 | column          |
| client_c14_mean_prior         | time_consistency   |         -0.1075 | column          |
| client_c3_mean_prior          | time_consistency   |         -0.1284 | column          |
| client_c4_mean_prior          | time_consistency   |         -0.1565 | column          |
| client_c7_mean_prior          | time_consistency   |         -0.1337 | column          |
| client_c8_mean_prior          | time_consistency   |         -0.1602 | column          |
| client_c9_mean_prior          | time_consistency   |         -0.1416 | column          |
| client_d10n_std_prior         | time_consistency   |         -0.1056 | column          |
| client_d11n_std_prior         | time_consistency   |         -0.0645 | column          |
| client_d15n_std_prior         | time_consistency   |         -0.1047 | column          |
| client_d2n_std_prior          | time_consistency   |         -0.1264 | column          |
| client_d5n_std_prior          | time_consistency   |         -0.0696 | column          |
| client_m2_mean_prior          | time_consistency   |         -0.111  | column          |
| client_m3_mean_prior          | time_consistency   |         -0.0931 | column          |
| client_m5_mean_prior          | time_consistency   |         -0.0765 | column          |
| id_14                         | time_consistency   |         -0.0886 | column          |
| id_38                         | time_consistency   |         -0.2426 | column          |

# Selection: PCA and forward selection

Two more ways to shrink a feature set, and what each one costs.

Code: [`src/fraud_detection/techniques/selection.py`](../../src/fraud_detection/techniques/selection.py)

Both were on this project's **not implemented** list, as score-chasing rather than
production safety. That reasoning still holds and is worth stating before the numbers:
[time consistency](time-consistency.md) rejects a feature because it will *break*; these
two drop a feature because it is not pulling its weight **today, on this holdout**. That
is a weaker claim, it does not transfer to next month's data, and a selection produced this
way should be re-derived when the data changes rather than pinned forever.

They are implemented because measuring them turned out to be cheap, and because one of the
two produced a result that confirms something else.

## PCA per family

The alternative to picking a representative: instead of keeping one column out of eleven,
keep a component built from all eleven.

| | |
| --- | ---: |
| Columns in | 339 |
| Components out | **129** |
| Runtime | 3 s |
| Explained variance, median | **0.949** |
| … minimum | 0.751 |
| Multi-column groups whose first component holds > 80% | **92 of 93 (99%)** |

**This is the strongest evidence that the families are real.** For 92 of 93 groups, a
single component reproduces almost all of the group's variance — which is what "these
columns are one signal" means, measured rather than asserted.

The exceptions are informative:

| Group | Columns | Explained variance |
| --- | ---: | ---: |
| `V108+V109+V110+V114` | 4 | **0.751** |
| `V170+V171+V200+V201` | 4 | 0.828 |
| `V186+V187+V190…+V199` | 8 | 0.836 |
| `V117+V118+V119` | 3 | 0.838 |

`V108+V109+V110+V114` is also the worst group in the
[correlation audit](redundancy.md#step-4-does-the-partition-hold). Two unrelated methods
flag the same group.

### What PCA costs, and why it is not the default

Nulls must be imputed before fitting — PCA has no notion of missing. In this dataset
missingness *is* signal, and a tree would have branched on it. Imputing it away discards
that. `explained` reports each group's null share so the cost is visible rather than
implied, but the cost is real: a component is never `NULL`, so "this family was absent for
this row" stops being expressible.

That is why representative selection ships as the default and PCA is the documented
alternative. One keeps a column a model can branch on and a human can name; the other keeps
more variance and neither.

## Forward selection

Grow a feature set one column at a time, keeping whichever column improves a holdout score
most, and stop when the improvement stops paying.

Run over the 129 representatives, first half of the time axis against the second, 60k rows
a side:

| | |
| --- | ---: |
| Candidates | 129 |
| Selected | **11** |
| Holdout ROC-AUC | **0.8222** |
| Runtime | 236 s on eight cores |
| Stopped because | gain +0.00049 fell below `min_gain` 0.001 |

| Step | Column | AUC | Gain |
| ---: | --- | ---: | ---: |
| 1 | `V11` | 0.6936 | +0.1936 |
| 2 | `V283` | 0.7451 | +0.0515 |
| 3 | `V258` | 0.7846 | +0.0394 |
| 4 | `V294` | 0.7904 | +0.0058 |
| 5 | `V86` | 0.8003 | +0.0099 |
| 6 | `V70` | 0.8054 | +0.0052 |
| 7 | `V284` | 0.8113 | +0.0059 |
| 8 | `V147` | 0.8170 | +0.0057 |
| 9 | `V338` | 0.8190 | +0.0020 |
| 10 | `V17` | 0.8204 | +0.0015 |
| 11 | `V171` | 0.8222 | +0.0018 |

Three columns reach 0.785 and the remaining eight buy 0.037 between them. The search
stopped on its own rather than hitting the cap, which is the outcome worth having: it means
the floor was doing the work, not the limit.

### `min_gain` is the honest part

A greedy search will always find *some* column that nudges a holdout score, so without a
floor it selects noise and reports it as improvement. The stopping reason is recorded in
the fragment so a reader can tell a search that converged from one that ran out of budget.

### Read this number carefully

**0.8222 is ROC-AUC on V columns alone**, on one holdout, and is not comparable to the
project's headline PR-AUC of 0.5046 over the full feature set. It says the V block carries
substantial signal in about a dozen columns. It does not say a model should be built from
eleven features.

### Cost

Quadratic: selecting *k* from *n* fits roughly *k·n* models. 129 candidates × 12 steps was
236 seconds parallelised. Unbounded forward selection over a few hundred columns is not a
check you run — it is a weekend. `max_features` and `min_gain` are required parameters for
that reason.

## Where these land

Both produce contract fragments, so a reduction becomes a decision the pipeline reads
rather than a number in a notebook. But of the five techniques these two are the ones whose
output should carry an expiry date: re-run them when the data changes, and do not treat
last quarter's selection as a fact about the problem.
